In [1]:
# STAGE 2 SETUP: RESTORE DATASET
# ------------------------------
import os
import zipfile
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define Paths (MUST MATCH STAGE 1)
PROJECT_PATH = '/content/drive/My Drive/Final Year Project'
ZIP_PATH = os.path.join(PROJECT_PATH, 'dataset.zip')
EXTRACT_ROOT = '/content/dataset_extraction' # The exact path Stage 1 used

# 3. Check and Re-Extract
if not os.path.exists(EXTRACT_ROOT):
    print(f"⚠️ Dataset missing (New Runtime Detected).")
    print(f"Extracting zip to {EXTRACT_ROOT} to match CSV paths...")

    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(EXTRACT_ROOT)
        print("✅ Extraction complete! The paths in your CSV are now valid.")
    else:
        print("❌ ERROR: Zip file not found in Drive. Check your paths!")
else:
    print("✅ Dataset already exists. Ready for training.")

# 4. Verify a random file
# (Optional) Checks if the specific file from your error exists now
test_path = '/content/dataset_extraction/dataset/train/Cotton healthy/h404.jpg'
# Note: Adjust 'dataset/train' if your zip structure is different
if os.path.exists(test_path):
    print(f"Verified: {test_path} exists.")
else:
    print("⚠️ Warning: Specific test file still missing. Check folder structure.")

Mounted at /content/drive
⚠️ Dataset missing (New Runtime Detected).
Extracting zip to /content/dataset_extraction to match CSV paths...
✅ Extraction complete! The paths in your CSV are now valid.
Verified: /content/dataset_extraction/dataset/train/Cotton healthy/h404.jpg exists.


In [2]:
# STAGE 2: MODEL TRAINING
# -----------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
from PIL import Image
from google.colab import drive
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

drive.mount('/content/drive')

# CONFIG
PROJECT_PATH = '/content/drive/My Drive/Final Year Project'
DATA_PATH = os.path.join(PROJECT_PATH, 'Processed_Data', 'full_dataset_metadata.csv')
MODEL_SAVE_PATH = os.path.join(PROJECT_PATH, 'Models')
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16 # Low batch size for big models on T4
NUM_EPOCHS = 5

# 1. Custom Dataset
# Replace your 'CropDataset' class with this one
class CropDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
        self.class_map = {name: i for i, name in enumerate(sorted(dataframe['label'].unique()))}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Try-Catch block to handle bad images safely
        try:
            row = self.df.iloc[idx]
            img = Image.open(row['filepath']).convert('RGB')
        except (IOError, OSError):
            # If image is broken, just pick the NEXT image instead of crashing
            # This is a common trick in ML pipelines
            new_idx = (idx + 1) % len(self.df)
            return self.__getitem__(new_idx)

        label = self.class_map[row['label']]

        if self.transform:
            img = self.transform(img)

        return img, label

# 2. Load Data
df = pd.read_csv(DATA_PATH)
num_classes = len(df['label'].unique())

# Transforms (Augmentation for train, plain for val)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']

train_loader = DataLoader(CropDataset(train_df, train_transforms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CropDataset(val_df, val_transforms), batch_size=BATCH_SIZE)

# 3. Model Definitions
def get_model(model_name, num_classes):
    if model_name == 'efficientnet':
        model = models.efficientnet_b5(weights='DEFAULT')
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == 'densenet':
        model = models.densenet201(weights='DEFAULT')
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == 'swin':
        # Using Swin-Tiny to fit in Colab memory. Swin-Base/Large might OOM.
        model = models.swin_t(weights='DEFAULT')
        model.head = nn.Linear(model.head.in_features, num_classes)
    return model.to(DEVICE)

# 4. Training Function
def train_and_save(model_name):
    print(f"--- Training {model_name} ---")
    model = get_model(model_name, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0.0

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        acc = 100 * correct / total
        print(f"Epoch {epoch+1}, Val Acc: {acc:.2f}%")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, f'{model_name}_best.pth'))

# 5. Execute Training (Run sequentially)
# Note: If Colab crashes, run one at a time by commenting others out
train_and_save('efficientnet')
train_and_save('densenet')
train_and_save('swin')

print("Stage 2 Complete. Models saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Training efficientnet ---
Downloading: "https://download.pytorch.org/models/efficientnet_b5_lukemelas-1a07897c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b5_lukemelas-1a07897c.pth


100%|██████████| 117M/117M [00:00<00:00, 140MB/s]


Epoch 1, Val Acc: 86.92%
Epoch 2, Val Acc: 89.18%
Epoch 3, Val Acc: 92.08%
Epoch 4, Val Acc: 94.66%
Epoch 5, Val Acc: 94.98%
--- Training densenet ---
Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth


100%|██████████| 77.4M/77.4M [00:00<00:00, 186MB/s]


Epoch 1, Val Acc: 86.99%
Epoch 2, Val Acc: 89.84%
Epoch 3, Val Acc: 91.94%
Epoch 4, Val Acc: 89.73%
Epoch 5, Val Acc: 93.98%
--- Training swin ---
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 132MB/s]


Epoch 1, Val Acc: 82.70%
Epoch 2, Val Acc: 86.40%
Epoch 3, Val Acc: 89.84%
Epoch 4, Val Acc: 90.96%
Epoch 5, Val Acc: 92.26%
Stage 2 Complete. Models saved.
